# Graph Feature Engineering

In [10]:
import pandas as pd
import numpy as np

leg_df = pd.read_csv(
    r"D:\Projects\Delivery_ETA\outputs\leg_level_data.csv"
)

node_metrics = pd.read_csv(
    r"D:\Projects\Delivery_ETA\outputs\node_metrics.csv"
)

corridor_metrics = pd.read_csv(
    r"D:\Projects\Delivery_ETA\outputs\corridor_metrics.csv"
)

print(leg_df.shape)
print(node_metrics.shape)
print(corridor_metrics.shape)

(26368, 12)
(1590, 5)
(2508, 6)


## Source Hub Features

In [11]:
source_features = node_metrics.add_prefix(
    "source_"
)

model_df = leg_df.merge(
    source_features,
    left_on="source_center",
    right_on="source_hub",
    how="left"
)

print(model_df.shape)

(26368, 17)


In [12]:
destination_features = node_metrics.add_prefix(
    "destination_"
)

model_df = model_df.merge(
    destination_features,
    left_on="destination_center",
    right_on="destination_hub",
    how="left"
)

print(model_df.shape)

(26368, 22)


## Corridor Features

In [13]:
corridor_features = corridor_metrics[
    [
        "source_center",
        "destination_center",
        "trip_count",
        "delay_ratio"
    ]
].copy()

corridor_features.columns = [
    "source_center",
    "destination_center",
    "corridor_trip_count",
    "corridor_delay_ratio"
]

model_df = model_df.merge(
    corridor_features,
    on=[
        "source_center",
        "destination_center"
    ],
    how="left"
)

print(model_df.shape)

(26368, 24)


In [14]:
print(model_df.shape)
print(model_df.columns.tolist())

(26368, 24)
['trip_uuid', 'source_center', 'destination_center', 'route_type', 'trip_creation_time', 'od_start_time', 'od_end_time', 'actual_time', 'osrm_time', 'osrm_distance', 'actual_distance_to_destination', 'data', 'source_hub', 'source_degree_centrality', 'source_betweenness_centrality', 'source_in_degree', 'source_out_degree', 'destination_hub', 'destination_degree_centrality', 'destination_betweenness_centrality', 'destination_in_degree', 'destination_out_degree', 'corridor_trip_count', 'corridor_delay_ratio']


## Temporal Features

In [15]:
date_cols = [
    "trip_creation_time",
    "od_start_time",
    "od_end_time"
]

for col in date_cols:
    model_df[col] = pd.to_datetime(
        model_df[col],
        format="mixed"
    )

print("Datetime Conversion Complete")

Datetime Conversion Complete


In [16]:
model_df["trip_hour"] = (
    model_df["trip_creation_time"]
    .dt.hour
)

model_df["trip_dayofweek"] = (
    model_df["trip_creation_time"]
    .dt.dayofweek
)

model_df["trip_month"] = (
    model_df["trip_creation_time"]
    .dt.month
)

model_df[
    [
        "trip_hour",
        "trip_dayofweek",
        "trip_month"
    ]
].head()

,trip_hour,trip_dayofweek,trip_month
0,0,2,9
1,0,2,9
2,0,2,9
3,0,2,9
4,0,2,9


In [17]:
model_df["route_type"] = (
    model_df["route_type"]
    .map({
        "FTL": 1,
        "Carting": 0
    })
)

print(
    model_df["route_type"]
    .value_counts()
)

route_type
1    13939
0    12429
Name: count, dtype: int64


In [18]:
model_df["distance_per_hour"] = (
    model_df["osrm_distance"]
    /
    model_df["osrm_time"]
)

model_df["network_risk_score"] = (
    model_df["corridor_delay_ratio"]
    *
    (
        model_df["source_betweenness_centrality"]
        +
        model_df["destination_betweenness_centrality"]
    )
)

model_df[
    [
        "distance_per_hour",
        "network_risk_score"
    ]
].describe()

,distance_per_hour,network_risk_score
count,26368.000000,25936.000000
mean,1.176067,0.087714
std,0.193383,0.152301
min,0.430468,0.000000
25%,1.043408,0.001010
50%,1.188563,0.014984
75%,1.350483,0.115363
max,1.634050,1.828936
